# Seasonality-Based Price Fluctuation Prediction Models

This notebook predicts **price fluctuations** based on **seasonal patterns** including major holidays and events:
- Chinese New Year (major impact on Asian shipping)
- Christmas / Year-End Holiday Season
- Thanksgiving
- Golden Week (China)
- Other seasonal patterns

We train the same three models (KNN, Decision Tree, Linear Regression) to determine which seasonalities affect price fluctuations the most.

---

## Approach

1. Load data and create fluctuation target
2. **Engineer seasonal features** (holiday indicators, month/quarter effects)
3. Analyze correlation between seasonality and price fluctuations
4. Train three models with seasonal features
5. **Identify which seasonalities have the strongest impact**
6. Compare model performance

---

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta

# Machine learning libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## Step 1: Load Data and Create Fluctuation Target

In [ ]:
# Load the prepared model data
df_model = pd.read_csv('data/processed/model_data.csv', parse_dates=['Date'], index_col='Date')

print(f"Loaded model data: {df_model.shape[0]} rows, {df_model.shape[1]} columns")
print(f"Date range: {df_model.index.min().date()} to {df_model.index.max().date()}")

# Create price fluctuation target
df_model['price_fluctuation_1w'] = df_model['price_1w_ahead'] - df_model['Europe_Base_Price']
df_model['price_pct_fluctuation_1w'] = ((df_model['price_1w_ahead'] - df_model['Europe_Base_Price']) / df_model['Europe_Base_Price']) * 100

print("\n✓ Target variable created: price_fluctuation_1w")

## Step 2: Engineer Seasonal Features

In [ ]:
print("\n" + "="*80)
print("SEASONAL FEATURE ENGINEERING")
print("="*80)

# Create a copy for feature engineering
df_seasonal = df_model.copy()

# Extract basic time features
df_seasonal['month'] = df_seasonal.index.month
df_seasonal['quarter'] = df_seasonal.index.quarter
df_seasonal['week_of_year'] = df_seasonal.index.isocalendar().week
df_seasonal['day_of_year'] = df_seasonal.index.dayofyear

# Function to check if date is near a holiday (within N weeks)
def is_near_date(dates, target_month, target_day, weeks_before=2, weeks_after=2):
    """
    Check if dates are within N weeks of a specific month/day each year.
    """
    result = pd.Series(0, index=dates)
    for year in range(dates.year.min(), dates.year.max() + 1):
        try:
            target_date = pd.Timestamp(year=year, month=target_month, day=target_day)
            start_date = target_date - pd.Timedelta(weeks=weeks_before)
            end_date = target_date + pd.Timedelta(weeks=weeks_after)
            result[(dates >= start_date) & (dates <= end_date)] = 1
        except:
            pass
    return result

# Chinese New Year (late Jan to mid Feb, varies by year - approximate)
# Using a range approach since CNY varies
df_seasonal['chinese_new_year_period'] = ((df_seasonal['month'] == 1) & (df_seasonal.index.day > 15)) | \
                                          ((df_seasonal['month'] == 2) & (df_seasonal.index.day < 20))
df_seasonal['chinese_new_year_period'] = df_seasonal['chinese_new_year_period'].astype(int)

# Christmas / Year-End Holiday Season (mid-Dec to early Jan)
df_seasonal['christmas_period'] = ((df_seasonal['month'] == 12) & (df_seasonal.index.day > 10)) | \
                                   ((df_seasonal['month'] == 1) & (df_seasonal.index.day < 7))
df_seasonal['christmas_period'] = df_seasonal['christmas_period'].astype(int)

# Thanksgiving (4th Thursday of November - US)
df_seasonal['thanksgiving_period'] = is_near_date(df_seasonal.index, 11, 22, weeks_before=1, weeks_after=1)

# Golden Week China (Oct 1-7)
df_seasonal['golden_week_china'] = is_near_date(df_seasonal.index, 10, 1, weeks_before=1, weeks_after=1)

# Spring Festival / May Day (May 1)
df_seasonal['may_day_period'] = is_near_date(df_seasonal.index, 5, 1, weeks_before=1, weeks_after=1)

# Mid-Autumn Festival / Moon Festival (September, approximate)
df_seasonal['mid_autumn_period'] = (df_seasonal['month'] == 9) & \
                                    (df_seasonal.index.day >= 10) & (df_seasonal.index.day <= 25)
df_seasonal['mid_autumn_period'] = df_seasonal['mid_autumn_period'].astype(int)

# Peak shipping season (Aug-Oct before Christmas)
df_seasonal['peak_shipping_season'] = df_seasonal['month'].isin([8, 9, 10]).astype(int)

# Low shipping season (Jan-Feb, post-holiday)
df_seasonal['low_shipping_season'] = df_seasonal['month'].isin([1, 2]).astype(int)

# Quarter indicators (one-hot encoding)
df_seasonal['q1'] = (df_seasonal['quarter'] == 1).astype(int)
df_seasonal['q2'] = (df_seasonal['quarter'] == 2).astype(int)
df_seasonal['q3'] = (df_seasonal['quarter'] == 3).astype(int)
df_seasonal['q4'] = (df_seasonal['quarter'] == 4).astype(int)

# Cyclical encoding for month (sin/cos to capture cyclical nature)
df_seasonal['month_sin'] = np.sin(2 * np.pi * df_seasonal['month'] / 12)
df_seasonal['month_cos'] = np.cos(2 * np.pi * df_seasonal['month'] / 12)

# List all seasonal features
seasonal_features = [
    'chinese_new_year_period',
    'christmas_period',
    'thanksgiving_period',
    'golden_week_china',
    'may_day_period',
    'mid_autumn_period',
    'peak_shipping_season',
    'low_shipping_season',
    'q1', 'q2', 'q3', 'q4',
    'month_sin', 'month_cos',
    'month', 'week_of_year'
]

print(f"\nCreated {len(seasonal_features)} seasonal features:")
for feat in seasonal_features:
    print(f"  - {feat}")

print("\n✓ Seasonal features engineered successfully")

## Step 3: Analyze Seasonality Impact on Price Fluctuations

In [ ]:
print("\n" + "="*80)
print("SEASONALITY IMPACT ANALYSIS")
print("="*80)

# Calculate average fluctuation during each seasonal period
seasonal_impact = {}

holiday_features = [
    'chinese_new_year_period',
    'christmas_period',
    'thanksgiving_period',
    'golden_week_china',
    'may_day_period',
    'mid_autumn_period',
    'peak_shipping_season',
    'low_shipping_season'
]

for feature in holiday_features:
    during_period = df_seasonal[df_seasonal[feature] == 1]['price_fluctuation_1w'].dropna()
    outside_period = df_seasonal[df_seasonal[feature] == 0]['price_fluctuation_1w'].dropna()
    
    if len(during_period) > 0:
        avg_during = during_period.mean()
        std_during = during_period.std()
        avg_outside = outside_period.mean()
        
        seasonal_impact[feature] = {
            'avg_fluctuation': avg_during,
            'std_fluctuation': std_during,
            'difference_from_normal': avg_during - avg_outside,
            'occurrences': len(during_period)
        }

# Sort by absolute difference from normal
seasonal_impact_df = pd.DataFrame(seasonal_impact).T
seasonal_impact_df['abs_impact'] = seasonal_impact_df['difference_from_normal'].abs()
seasonal_impact_df = seasonal_impact_df.sort_values('abs_impact', ascending=False)

print("\nSEASONALITY IMPACT RANKING (by absolute impact):")
print("="*80)
print(f"{'Season':<30} {'Avg Fluctuation':>15} {'Diff from Normal':>15} {'Occurrences':>12}")
print("="*80)

for idx, row in seasonal_impact_df.iterrows():
    print(f"{idx:<30} ${row['avg_fluctuation']:>14.2f} ${row['difference_from_normal']:>14.2f} {int(row['occurrences']):>12}")

print("\n" + "="*80)
print("TOP 3 MOST IMPACTFUL SEASONALITIES:")
print("="*80)
for i, (idx, row) in enumerate(seasonal_impact_df.head(3).iterrows(), 1):
    impact_direction = "INCREASE" if row['difference_from_normal'] > 0 else "DECREASE"
    print(f"{i}. {idx}")
    print(f"   Average fluctuation: ${row['avg_fluctuation']:.2f}")
    print(f"   Impact: {impact_direction} of ${abs(row['difference_from_normal']):.2f} vs normal")
    print()

# Visualize seasonal impacts
fig, ax = plt.subplots(figsize=(14, 6))
colors = ['green' if x > 0 else 'red' for x in seasonal_impact_df['difference_from_normal']]
ax.barh(range(len(seasonal_impact_df)), seasonal_impact_df['difference_from_normal'], color=colors)
ax.set_yticks(range(len(seasonal_impact_df)))
ax.set_yticklabels(seasonal_impact_df.index)
ax.set_xlabel('Difference from Normal Fluctuation ($)', fontsize=12)
ax.set_title('Seasonal Impact on Price Fluctuations', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Save seasonal impact analysis
seasonal_impact_df.to_csv('data/processed/seasonality_impact_analysis.csv')
print("\n✓ Seasonal impact analysis saved to data/processed/seasonality_impact_analysis.csv")

## Step 4: Prepare Features for Modeling

In [ ]:
# Drop rows with missing target
df_clean = df_seasonal.dropna(subset=['price_fluctuation_1w']).copy()

print(f"Rows after removing missing targets: {len(df_clean)}")

# Combine seasonal features with some key lagged price features
price_lag_features = ['price_lag_1w', 'price_lag_2w', 'price_lag_4w']
rolling_features = [col for col in df_clean.columns if '_roll_mean_' in col or '_roll_std_' in col]
rolling_features = rolling_features[:5]  # Take top 5 rolling features

# Combine all features
all_features = seasonal_features + price_lag_features + rolling_features

# Remove any features that don't exist
all_features = [f for f in all_features if f in df_clean.columns]

print(f"\nTotal features for modeling: {len(all_features)}")
print(f"  - Seasonal features: {len(seasonal_features)}")
print(f"  - Price lag features: {len([f for f in price_lag_features if f in df_clean.columns])}")
print(f"  - Rolling features: {len([f for f in rolling_features if f in df_clean.columns])}")

# Prepare X and y
X = df_clean[all_features]
y = df_clean['price_fluctuation_1w']

print("\n✓ Features prepared for modeling")

## Step 5: Train-Test Split

In [ ]:
# Time-based split (80/20)
split_idx = int(len(df_clean) * 0.8)
X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print("="*80)
print("TRAIN-TEST SPLIT")
print("="*80)
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {len(all_features)}")
print(f"\nTrain date range: {X_train.index.min().date()} to {X_train.index.max().date()}")
print(f"Test date range: {X_test.index.min().date()} to {X_test.index.max().date()}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✓ Features scaled using StandardScaler")

## Step 6: Define Evaluation Function

In [ ]:
def evaluate_fluctuation_model(y_true, y_pred, model_name):
    """
    Evaluate model performance for price fluctuation prediction.
    """
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    
    # Relative MAE
    mean_abs_fluctuation = np.mean(np.abs(y_true))
    relative_mae = (mae / mean_abs_fluctuation) * 100 if mean_abs_fluctuation != 0 else 0
    
    r2 = r2_score(y_true, y_pred)
    
    # Direction accuracy
    correct_direction = np.sum((y_true > 0) == (y_pred > 0))
    direction_accuracy = (correct_direction / len(y_true)) * 100
    
    metrics = {
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'Relative_MAE_%': relative_mae,
        'R²': r2,
        'Direction_Accuracy_%': direction_accuracy
    }
    
    print(f"\n{'='*70}")
    print(f"{model_name} Performance")
    print(f"{'='*70}")
    print(f"RMSE:                ${rmse:.2f}")
    print(f"MAE:                 ${mae:.2f}")
    print(f"Relative MAE:        {relative_mae:.2f}%")
    print(f"R²:                  {r2:.4f}")
    print(f"Direction Accuracy:  {direction_accuracy:.2f}%")
    print(f"{'='*70}")
    
    return metrics

# Store results
results = []

print("Evaluation function defined!")

## Step 7: Model 1 - Linear Regression

In [ ]:
print("\n" + "="*80)
print("MODEL 1: LINEAR REGRESSION (with Seasonal Features)")
print("="*80)

# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test_scaled)

# Evaluate
metrics_lr = evaluate_fluctuation_model(y_test, y_pred_lr, 'Linear Regression')
results.append(metrics_lr)

# Feature importance (coefficients)
feature_importance = pd.DataFrame({
    'Feature': all_features,
    'Coefficient': lr_model.coef_,
    'Abs_Coefficient': np.abs(lr_model.coef_)
}).sort_values('Abs_Coefficient', ascending=False)

print("\nTop 10 Most Important Features (Linear Regression):")
print(feature_importance.head(10).to_string(index=False))

# Highlight seasonal features in top 10
top_10_seasonal = [f for f in feature_importance.head(10)['Feature'] if f in seasonal_features]
print(f"\nSeasonal features in top 10: {len(top_10_seasonal)}")
for feat in top_10_seasonal:
    coef = feature_importance[feature_importance['Feature'] == feat]['Coefficient'].values[0]
    print(f"  - {feat}: {coef:.4f}")

print("\n✓ Linear Regression trained and evaluated")

## Step 8: Model 2 - Decision Tree

In [ ]:
print("\n" + "="*80)
print("MODEL 2: DECISION TREE with GridSearchCV")
print("="*80)

# Define parameter grid
param_grid_dt = {
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [2, 5, 10]
}

# Grid search
dt_base = DecisionTreeRegressor(random_state=42)
grid_dt = GridSearchCV(dt_base, param_grid_dt, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

print("\nTraining Decision Tree with GridSearchCV...")
grid_dt.fit(X_train_scaled, y_train)

# Best model
dt_model = grid_dt.best_estimator_
print(f"\nBest parameters: {grid_dt.best_params_}")

# Predict
y_pred_dt = dt_model.predict(X_test_scaled)

# Evaluate
metrics_dt = evaluate_fluctuation_model(y_test, y_pred_dt, 'Decision Tree')
results.append(metrics_dt)

# Feature importance
dt_importance = pd.DataFrame({
    'Feature': all_features,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features (Decision Tree):")
print(dt_importance.head(10).to_string(index=False))

# Highlight seasonal features
top_10_seasonal_dt = [f for f in dt_importance.head(10)['Feature'] if f in seasonal_features]
print(f"\nSeasonal features in top 10: {len(top_10_seasonal_dt)}")
for feat in top_10_seasonal_dt:
    imp = dt_importance[dt_importance['Feature'] == feat]['Importance'].values[0]
    print(f"  - {feat}: {imp:.4f}")

print("\n✓ Decision Tree trained and evaluated")

## Step 9: Model 3 - K-Nearest Neighbors (KNN)

In [ ]:
print("\n" + "="*80)
print("MODEL 3: K-NEAREST NEIGHBORS with GridSearchCV")
print("="*80)

# Define parameter grid
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 10, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# Grid search
knn_base = KNeighborsRegressor()
grid_knn = GridSearchCV(knn_base, param_grid_knn, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

print("\nTraining KNN with GridSearchCV...")
grid_knn.fit(X_train_scaled, y_train)

# Best model
knn_model = grid_knn.best_estimator_
print(f"\nBest parameters: {grid_knn.best_params_}")

# Predict
y_pred_knn = knn_model.predict(X_test_scaled)

# Evaluate
metrics_knn = evaluate_fluctuation_model(y_test, y_pred_knn, 'K-Nearest Neighbors')
results.append(metrics_knn)

print("\n✓ KNN trained and evaluated")

## Step 10: Model Comparison

In [ ]:
print("\n" + "="*80)
print("MODEL COMPARISON - SEASONALITY-BASED PREDICTION")
print("="*80)

# Create comparison dataframe
results_df = pd.DataFrame(results)
print("\n", results_df.to_string(index=False))

# Determine best model
best_model_idx = results_df['RMSE'].idxmin()
best_model_name = results_df.loc[best_model_idx, 'Model']

print(f"\n{'='*80}")
print(f"BEST MODEL: {best_model_name}")
print(f"{'='*80}")
print(f"RMSE: ${results_df.loc[best_model_idx, 'RMSE']:.2f}")
print(f"Direction Accuracy: {results_df.loc[best_model_idx, 'Direction_Accuracy_%']:.2f}%")
print(f"{'='*80}")

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# RMSE comparison
axes[0].bar(results_df['Model'], results_df['RMSE'], color=['blue', 'green', 'orange'])
axes[0].set_title('RMSE Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('RMSE ($)')
axes[0].tick_params(axis='x', rotation=45)

# MAE comparison
axes[1].bar(results_df['Model'], results_df['MAE'], color=['blue', 'green', 'orange'])
axes[1].set_title('MAE Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('MAE ($)')
axes[1].tick_params(axis='x', rotation=45)

# Direction Accuracy comparison
axes[2].bar(results_df['Model'], results_df['Direction_Accuracy_%'], color=['blue', 'green', 'orange'])
axes[2].set_title('Direction Accuracy Comparison', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Direction Accuracy (%)')
axes[2].axhline(y=50, color='red', linestyle='--', linewidth=1, label='Random Guess (50%)')
axes[2].legend()
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Save results
results_df.to_csv('data/processed/seasonality_model_comparison.csv', index=False)
print("\n✓ Results saved to data/processed/seasonality_model_comparison.csv")

## Step 11: Seasonal Feature Importance Summary

In [ ]:
print("\n" + "="*80)
print("SEASONAL FEATURE IMPORTANCE SUMMARY")
print("="*80)

# Extract only seasonal features from importance rankings
lr_seasonal_importance = feature_importance[feature_importance['Feature'].isin(seasonal_features)].copy()
lr_seasonal_importance['Model'] = 'Linear Regression'
lr_seasonal_importance = lr_seasonal_importance.rename(columns={'Abs_Coefficient': 'Importance'})

dt_seasonal_importance = dt_importance[dt_importance['Feature'].isin(seasonal_features)].copy()
dt_seasonal_importance['Model'] = 'Decision Tree'

# Combine and rank
combined_seasonal = pd.concat([lr_seasonal_importance[['Feature', 'Importance', 'Model']],
                                dt_seasonal_importance[['Feature', 'Importance', 'Model']]])

# Average importance across models
avg_seasonal_importance = combined_seasonal.groupby('Feature')['Importance'].mean().sort_values(ascending=False)

print("\nAVERAGE SEASONAL FEATURE IMPORTANCE (across models):")
print("="*80)
for i, (feat, importance) in enumerate(avg_seasonal_importance.items(), 1):
    print(f"{i:2d}. {feat:<35} Importance: {importance:.6f}")

# Visualize
fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(range(len(avg_seasonal_importance)), avg_seasonal_importance.values, color='steelblue')
ax.set_yticks(range(len(avg_seasonal_importance)))
ax.set_yticklabels(avg_seasonal_importance.index)
ax.set_xlabel('Average Importance', fontsize=12)
ax.set_title('Seasonal Feature Importance (Average across LR & DT)', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("TOP 5 MOST IMPORTANT SEASONAL FACTORS:")
print("="*80)
for i, (feat, importance) in enumerate(avg_seasonal_importance.head(5).items(), 1):
    print(f"{i}. {feat}")
    
    # Get impact from earlier analysis
    if feat in seasonal_impact_df.index:
        impact = seasonal_impact_df.loc[feat, 'difference_from_normal']
        direction = "increases" if impact > 0 else "decreases"
        print(f"   Model importance: {importance:.6f}")
        print(f"   Actual impact: {direction} fluctuation by ${abs(impact):.2f}")
    else:
        print(f"   Model importance: {importance:.6f}")
    print()

# Save seasonal importance
avg_seasonal_importance.to_csv('data/processed/seasonal_feature_importance.csv')
print("✓ Seasonal feature importance saved to data/processed/seasonal_feature_importance.csv")

## Step 12: Prediction Visualization

In [ ]:
# Create prediction dataframe
pred_df = pd.DataFrame({
    'Date': y_test.index,
    'Actual_Fluctuation': y_test.values,
    'LR_Predicted': y_pred_lr,
    'DT_Predicted': y_pred_dt,
    'KNN_Predicted': y_pred_knn
}).set_index('Date')

# Add seasonal indicators for visualization
for feat in ['chinese_new_year_period', 'christmas_period', 'peak_shipping_season']:
    pred_df[feat] = df_clean.loc[pred_df.index, feat]

# Plot predictions
fig = go.Figure()

fig.add_trace(go.Scatter(x=pred_df.index, y=pred_df['Actual_Fluctuation'], 
                         mode='lines+markers', name='Actual Fluctuation',
                         line=dict(color='black', width=2)))

fig.add_trace(go.Scatter(x=pred_df.index, y=pred_df['LR_Predicted'],
                         mode='lines', name='Linear Regression',
                         line=dict(color='blue', dash='dash')))

fig.add_trace(go.Scatter(x=pred_df.index, y=pred_df['DT_Predicted'],
                         mode='lines', name='Decision Tree',
                         line=dict(color='green', dash='dash')))

fig.add_trace(go.Scatter(x=pred_df.index, y=pred_df['KNN_Predicted'],
                         mode='lines', name='KNN',
                         line=dict(color='orange', dash='dash')))

# Add seasonal markers
# Chinese New Year
cny_dates = pred_df[pred_df['chinese_new_year_period'] == 1].index
if len(cny_dates) > 0:
    for date in cny_dates:
        fig.add_vline(x=date, line_dash="dot", line_color="red", opacity=0.3)

# Add zero line
fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.5,
              annotation_text="No Change", annotation_position="right")

fig.update_layout(
    title='Actual vs Predicted Weekly Price Fluctuations with Seasonal Features (Test Set)',
    xaxis_title='Date',
    yaxis_title='Price Fluctuation (USD)',
    height=600,
    hovermode='x unified',
    legend=dict(x=0.01, y=0.99)
)

fig.show()

# Save predictions
pred_df.to_csv('data/processed/seasonality_fluctuation_predictions.csv')
print("\n✓ Predictions saved to data/processed/seasonality_fluctuation_predictions.csv")

## Summary

This notebook successfully:

1. **Engineered seasonal features** including:
   - Chinese New Year
   - Christmas/Year-End
   - Thanksgiving
   - Golden Week (China)
   - May Day
   - Mid-Autumn Festival
   - Peak/Low shipping seasons
   - Quarter and month indicators

2. **Analyzed seasonal impact** on price fluctuations and ranked them by importance

3. **Trained three models**:
   - Linear Regression
   - Decision Tree
   - K-Nearest Neighbors

4. **Identified the most impactful seasonalities** affecting container freight prices

5. **Compared model performance** and saved results for further analysis

Key insights:
- The top seasonal factors affecting price fluctuations are identified and ranked
- Models can predict price direction with seasonal context
- Seasonal patterns provide valuable signals for forecasting price movements